# Лабораторная работа № 3.2
## Гипотеза о равенстве дисперсий

**Выполнили:**
- Левченков Артемий Викторович;
- Бикбулатова Динара Фануровна;
- Ладыгин Максим Александрович.

**Группа:** БИВТ-24-АЭ-3

**Исходные данные:** две случайные повторные выборки значений возраста из файла `data/moscow.txt`.

**Цель работы:** при уровне значимости $\alpha=0{,}05$ проверить гипотезу $H_0:D_1=D_2$ при конкурирующих гипотезах: а) $H_1:D_1>D_2$; б) $H_1:D_1\ne D_2$.

В ходе исследования рассчитываются исправленные дисперсии, объёмы выборок, числа степеней свободы, наблюдаемое и критические значения F-критерия.

### 1. Исходные данные и объём выборки

Среднее и дисперсия исходной совокупности рассчитываются по дискретному ряду:

$$\bar x=\frac{\sum x_i n_i}{\sum n_i},\qquad D=\frac{\sum(x_i-\bar x)^2n_i}{\sum n_i}.$$

Объём повторной выборки определяется по формуле:

$$n=\left\lceil\frac{t^2\sigma^2}{\delta^2}\right\rceil,\qquad 2\Phi(t)=\gamma.$$

Принимаются $\gamma=0{,}95$, $t=F_{N(0,1)}^{-1}((1+\gamma)/2)\approx1{,}96$, $\delta=3$ года. Здесь $\sigma^2=D$ — дисперсия всей исходной совокупности. $\Phi$ обозначает функцию Лапласа.

Коэффициент `t` вычисляется с помощью `NormalDist().inv_cdf((1 + gamma) / 2)`.

In [1]:
import pandas as pd
from math import ceil
from statistics import NormalDist
from scipy.stats import f

data = pd.read_table("data/moscow.txt", header=None)[0]
N = len(data)

print("Количество наблюдений:", N)
print("Минимальный возраст:", data.min())
print("Максимальный возраст:", data.max())

Количество наблюдений: 32423
Минимальный возраст: 14
Максимальный возраст: 73


In [2]:
def mean_value(x, n):
    return sum(x * n) / sum(n)


def dispersion(x, n, mean):
    return sum((x - mean) ** 2 * n) / sum(n)


discrete = data.value_counts().sort_index().rename_axis("x").reset_index(name="n")
x = discrete["x"]
n = discrete["n"]

mean_age = mean_value(x, n)
D_age = dispersion(x, n, mean_age)

gamma = 0.95
delta = 3
t = NormalDist().inv_cdf((1 + gamma) / 2)
n_exact = t ** 2 * D_age / delta ** 2
sample_size = ceil(n_exact)

print(f"Средний возраст: {mean_age:.4f} года")
print(f"Дисперсия исходной совокупности: {D_age:.4f} года²")
print(f"Надёжность γ = {gamma}, точность δ = {delta} года, t = {t}")
print(f"Расчётный объём выборки: {n_exact:.4f}")
print(f"Объём каждой выборки: {sample_size}")

Средний возраст: 35.3730 года
Дисперсия исходной совокупности: 144.9170 года²
Надёжность γ = 0.95, точность δ = 3 года, t = 1.9599639845400534
Расчётный объём выборки: 61.8547
Объём каждой выборки: 62


### 2. Две выборки и исправленные дисперсии

Исправленная выборочная дисперсия определяется по формуле:

$$s^2=\frac{n}{n-1}D_{\text{в}}=\frac{1}{n-1}\sum_{j=1}^{n}(x_j-\bar x)^2.$$

Выборочная дисперсия $D_{\text{в}}$ умножается на поправку Бесселя $n/(n-1)$.

Из исходной совокупности независимо формируются две случайные выборки значений возраста с возвращением. При повторном запуске выполняется новый отбор. Перед проверкой гипотез дисперсии упорядочиваются по убыванию.

In [3]:
sample1 = data.sample(n=sample_size, replace=True)
sample2 = data.sample(n=sample_size, replace=True)

samples = [sample1, sample2]
rows = []

for i in range(len(samples)):
    sample = samples[i]
    n_sample = len(sample)
    sample_discrete = sample.value_counts().sort_index().rename_axis("x").reset_index(name="n")
    x = sample_discrete["x"]
    n = sample_discrete["n"]

    sample_mean = mean_value(x, n)
    D_sample = dispersion(x, n, sample_mean)
    s_squared = n_sample / (n_sample - 1) * D_sample

    rows.append([i + 1, n_sample, sample_mean, D_sample, s_squared, n_sample - 1])

sample_table = pd.DataFrame(rows, columns=["Выборка", "n", "Средняя", "D_в", "s²", "k"])
display(sample_table.round(4))

,Выборка,n,Средняя,D_в,s²,k
0,1,62,35.4355,103.7297,105.4302,61
1,2,62,36.5161,189.3788,192.4833,61


### 3. Наблюдаемое значение F-критерия

Наблюдаемое значение критерия рассчитывается как отношение большей исправленной дисперсии к меньшей:

$$F_{\text{набл}}=\frac{s_1^2}{s_2^2},\qquad s_1^2\ge s_2^2,\qquad k_1=n_1-1,\qquad k_2=n_2-1.$$

Индекс 1 присваивается выборке с большей исправленной дисперсией, индекс 2 — с меньшей. Вместе с дисперсиями переставляются соответствующие объёмы выборок. Поэтому $F_{\text{набл}}\ge1$.

При независимых выборках из нормальных совокупностей и справедливости $H_0$ отношение исправленных дисперсий в фиксированном порядке имеет распределение Фишера–Снедекора. В расчёте используется упорядоченное отношение большей дисперсии к меньшей.

In [4]:
alpha = 0.05
n1 = len(sample1)
n2 = len(sample2)
s1_2 = sample_table.loc[0, "s²"]
s2_2 = sample_table.loc[1, "s²"]
if s1_2 < s2_2:
    s1_2, s2_2 = s2_2, s1_2
    n1, n2 = n2, n1

k1 = n1 - 1
k2 = n2 - 1
F_obs = s1_2 / s2_2

print(f"Исправленная дисперсия выборки 1 (большая): s₁² = {s1_2:.4f} года²")
print(f"Исправленная дисперсия выборки 2 (меньшая): s₂² = {s2_2:.4f} года²")
print(f"Объёмы выборок: n₁ = {n1}, n₂ = {n2}")
print(f"Степени свободы: k₁ = {k1}, k₂ = {k2}")
print(f"Уровень значимости: α = {alpha}")
print(f"F наблюдаемое = {s1_2:.4f} / {s2_2:.4f} = {F_obs:.4f}")

Исправленная дисперсия выборки 1 (большая): s₁² = 192.4833 года²
Исправленная дисперсия выборки 2 (меньшая): s₂² = 105.4302 года²
Объёмы выборок: n₁ = 62, n₂ = 62
Степени свободы: k₁ = 61, k₂ = 61
Уровень значимости: α = 0.05
F наблюдаемое = 192.4833 / 105.4302 = 1.8257


### 4. а) Конкурирующая гипотеза $H_1:D_1>D_2$

Правосторонняя критическая область определяется условием:

$$P(F>F_{\text{кр}}\mid H_0)=\alpha,\qquad F_{\text{кр}}=Q_{1-\alpha}(k_1,k_2).$$

$Q_p$ — квантиль с вероятностью $p$ слева от него. Квантили рассчитываются с помощью `f.ppf(p, k1, k2)`. Правой критической точке $F_{\text{кр}}(\alpha;k_1;k_2)$ соответствует `f.ppf(1 - alpha, k1, k2)`.

Если $F_{\text{набл}}>F_{\text{кр}}$, отвергаем $H_0$ в пользу $D_1>D_2$. Иначе оснований отвергать $H_0$ нет.

Здесь индексы 1 и 2 относятся к выборкам после упорядочивания по величине исправленной дисперсии.

In [5]:
F_crit = f.ppf(1 - alpha, k1, k2)

print("а) H₁: D₁ > D₂")
print(f"F наблюдаемое = {F_obs:.4f}")
print(f"F критическое = {F_crit:.4f}")
print(f"Критическая область: F > {F_crit:.4f}")

if F_obs > F_crit:
    print("H₀ отвергается: дисперсия совокупности 1 значимо больше дисперсии совокупности 2.")
else:
    print("Оснований отвергать H₀ нет: превышение D₁ над D₂ не установлено.")

а) H₁: D₁ > D₂
F наблюдаемое = 1.8257
F критическое = 1.5288
Критическая область: F > 1.5288
H₀ отвергается: дисперсия совокупности 1 значимо больше дисперсии совокупности 2.


### 5. б) Конкурирующая гипотеза $H_1:D_1\ne D_2$

Для двусторонней критической области на каждый хвост приходится вероятность $\alpha/2$.

Поскольку большая исправленная дисперсия находится в числителе, используется правая критическая точка:

$$F_{\text{кр}}=Q_{1-\alpha/2}(k_1,k_2).$$

Отвергаем $H_0$, если $F_{\text{набл}}>F_{\text{кр}}$. Иначе оснований отвергать $H_0$ нет. Степени свободы $k_1$ и $k_2$ соответствуют выборкам в числителе и знаменателе.

In [6]:
F_crit_two = f.ppf(1 - alpha / 2, k1, k2)

print("б) H₁: D₁ ≠ D₂")
print(f"F наблюдаемое = {F_obs:.4f}")
print(f"F критическое = {F_crit_two:.4f}")
print(f"Критическая область: F > {F_crit_two:.4f}")

if F_obs > F_crit_two:
    print("H₀ отвергается: дисперсии генеральных совокупностей значимо различаются.")
else:
    print("Оснований отвергать H₀ нет: значимого различия дисперсий не обнаружено.")

б) H₁: D₁ ≠ D₂
F наблюдаемое = 1.8257
F критическое = 1.6597
Критическая область: F > 1.6597
H₀ отвергается: дисперсии генеральных совокупностей значимо различаются.


### 6. Вывод

In [7]:
print(f"Из исходной совокупности сформированы две повторные выборки по {sample_size} наблюдения.")
print(f"После упорядочивания: s₁² = {s1_2:.4f}, s₂² = {s2_2:.4f} года².")
print(f"Степени свободы: k₁ = {k1}, k₂ = {k2}; F наблюдаемое = {F_obs:.4f}.")

if F_obs > F_crit:
    print(f"а) F > {F_crit:.4f}: H₀ отвергается в пользу D₁ > D₂ при α = {alpha}.")
else:
    print(f"а) F ≤ {F_crit:.4f}: оснований отвергать H₀ при α = {alpha} нет.")

if F_obs > F_crit_two:
    print(f"б) F > {F_crit_two:.4f}: H₀ отвергается при α = {alpha}.")
else:
    print(f"б) F ≤ {F_crit_two:.4f}: оснований отвергать H₀ при α = {alpha} нет.")

print("Неотвержение H₀ не доказывает точное равенство дисперсий.")

Из исходной совокупности сформированы две повторные выборки по 62 наблюдения.
После упорядочивания: s₁² = 192.4833, s₂² = 105.4302 года².
Степени свободы: k₁ = 61, k₂ = 61; F наблюдаемое = 1.8257.
а) F > 1.5288: H₀ отвергается в пользу D₁ > D₂ при α = 0.05.
б) F > 1.6597: H₀ отвергается при α = 0.05.
Неотвержение H₀ не доказывает точное равенство дисперсий.


### Условия применения и интерпретация

Обе выборки получены независимым случайным отбором из одной исходной совокупности, поэтому генеральная дисперсия для них одинакова. Различия выборочных дисперсий обусловлены случайностью отбора.

Точное F-распределение отношения дисперсий предполагает **нормальность исходных генеральных совокупностей**. Нормальность выборочных средних не означает нормальность отдельных возрастов. Полученные результаты интерпретируются при предположении нормальности исходного распределения. При нарушении этого условия фактическая вероятность ошибки первого рода может отличаться от заданного уровня значимости 0,05.

В пункте а) направление сравнения задаётся после упорядочивания выборочных дисперсий. Поэтому этот расчёт не является проверкой заранее заданного направления между исходными выборками.